# 5200 Final Exam — Mock Set 3 (with Answer Code)
## Dataset: Airline Passenger Satisfaction (`airline_satisfaction.csv`)

### Exam Rules
- **Time**: 1 hr 50 min | `alpha = 0.05` | **`random_state = 1031`** ⚠️
- Do NOT round | No commas | Lead with 0 | Open-book

### Modules Covered
Non-Linear Regression · Feature Selection · Trees (Regression) · Ensemble (Regression) · SVR

### Variables
| Variable | Description |
|---|---|
| `Satisfaction` | **Outcome** (numeric 1–5) |
| `Age` | Passenger age |
| `Gender` | Male/Female/Other |
| `FrequentFlyer` | Yes/No |
| `FlightDistance` | Miles |
| `DepartureDelay` | Departure delay (min) |
| `ArrivalDelay` | Arrival delay (min) |
| `SeatComfort` | Rating 1–5 |
| `InFlightEntertainment` | Rating 1–5 |
| `FoodService` | Rating 1–5 |
| `OnboardService` | Rating 1–5 |
| `BaggageHandling` | Rating 1–5 |
| `CheckinEase` | Rating 1–5 |
| `WifiConnectivity` | Rating 1–5 |

> **Note**: This set uses `random_state = 1031` and focuses on **regression** (numeric outcome).

---
## ❓ Q1. [Dataset Confirmation]
> Read in `airline_satisfaction.csv`. How many rows and columns?

In [ ]:
import pandas as pd
import numpy as np

data = pd.read_csv('airline_satisfaction.csv')
print('Shape:', data.shape)
print(data.dtypes)

---
## ❓ Q2. [Setup]
> Assign `Satisfaction` to `y`. Assign all other columns to `X`.
> Use stratified split: 70% train, 30% test, `random_state=1031`.
> (Stratify by `Satisfaction` directly.)
> **What is the MEDIAN of `Satisfaction` in the TRAIN sample?**

In [ ]:
from sklearn.model_selection import train_test_split

y = data.Satisfaction
X = data.drop('Satisfaction', axis=1)

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, train_size=0.7, random_state=1031, stratify=y)

print('Train shape:', X_train_raw.shape)
print('Median Satisfaction in train:', y_train.median())

---
## Section 1: Non-Linear Regression
*(Predict `Satisfaction` from `FlightDistance`. Use train/test split from Q2.)*

---
## ❓ Q3.
> Fit three models predicting `Satisfaction` from `FlightDistance`:
> - `model_lin`:   degree 1
> - `model_poly2`: degree 2
> - `model_poly3`: degree 3
>
> For `model_poly2`, **is the quadratic term significant? What is its p-value?**

In [ ]:
from statsmodels.formula.api import ols

train_df = X_train_raw.copy(); train_df['Satisfaction'] = y_train.values
test_df  = X_test_raw.copy();  test_df['Satisfaction']  = y_test.values

model_lin   = ols('Satisfaction ~ FlightDistance', data=train_df).fit()
model_poly2 = ols('Satisfaction ~ FlightDistance + I(FlightDistance**2)', data=train_df).fit()
model_poly3 = ols('Satisfaction ~ FlightDistance + I(FlightDistance**2) + I(FlightDistance**3)', data=train_df).fit()

print('Poly-2 summary:')
print(model_poly2.summary())
print('\np-value of quadratic term:', model_poly2.pvalues['I(FlightDistance ** 2)'])

---
## ❓ Q4.
> Create a step function:
> - Bins: `[0, 1000, 2500, 5000, inf)` → `short, medium, long, ultra`
> - Fit: `Satisfaction ~ dist_bin`
>
> **What is the coefficient for `ultra` flights compared to `short`?**

In [ ]:
train_df = train_df.copy(); test_df = test_df.copy()

bins   = [0, 1000, 2500, 5000, float('inf')]
labels = ['short', 'medium', 'long', 'ultra']

train_df['dist_bin'] = pd.cut(train_df.FlightDistance, bins=bins, labels=labels)
test_df['dist_bin']  = pd.cut(test_df.FlightDistance,  bins=bins, labels=labels)

model_step = ols('Satisfaction ~ dist_bin', data=train_df).fit()
print(model_step.summary())
print('\nCoefficients (vs reference=short):')
print(model_step.params)
print('\nUltra vs Short:', model_step.params['dist_bin[T.ultra]'])

---
## ❓ Q5.
> Compare test RMSE for all four models: `model_lin`, `model_poly2`, `model_poly3`, `model_step`.
> **Which model has the lowest test RMSE?**

In [ ]:
def rmse(y_true, y_pred):
    return np.sqrt(np.mean((np.array(y_true) - np.array(y_pred)) ** 2))

results = pd.DataFrame({
    'model': ['linear','poly2','poly3','step'],
    'train_rmse': [
        rmse(train_df.Satisfaction, model_lin.predict()),
        rmse(train_df.Satisfaction, model_poly2.predict()),
        rmse(train_df.Satisfaction, model_poly3.predict()),
        rmse(train_df.Satisfaction, model_step.predict()),
    ],
    'test_rmse': [
        rmse(test_df.Satisfaction, model_lin.predict(test_df)),
        rmse(test_df.Satisfaction, model_poly2.predict(test_df)),
        rmse(test_df.Satisfaction, model_poly3.predict(test_df)),
        rmse(test_df.Satisfaction, model_step.predict(test_df)),
    ]
}).set_index('model')
print(results)
print('\nBest test RMSE:', results.test_rmse.idxmin())

---
## ❓ Q6.
> Using sklearn `PolynomialFeatures(degree=2)` on `FlightDistance` only,
> fit a `LinearRegression`.
> **What is the predicted `Satisfaction` for `FlightDistance = 2000`?**

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression

poly = PolynomialFeatures(degree=2, include_bias=False)
X_tr_p = poly.fit_transform(X_train_raw[['FlightDistance']])
X_te_p = poly.transform(X_test_raw[['FlightDistance']])

lr_poly = LinearRegression().fit(X_tr_p, y_train)

# Predict for FlightDistance = 2000
new_obs = poly.transform([[2000]])
print('Predicted Satisfaction for FlightDistance=2000:', lr_poly.predict(new_obs)[0])

---
## Section 2: Feature Selection
*(Predict `Satisfaction` using all features. Dummy-encode `Gender` and `FrequentFlyer`.)*

---
## ❓ Q7.
> Dummy-encode `Gender` and `FrequentFlyer` (drop_first=True). Align test to train.
> Compute Pearson correlations with `Satisfaction`.
> **Which feature has the STRONGEST (highest |r|) correlation?**

In [ ]:
from scipy.stats import pearsonr

cat_cols = ['Gender', 'FrequentFlyer']
X_tr = pd.get_dummies(X_train_raw, columns=cat_cols, drop_first=True).astype(float)
X_te = pd.get_dummies(X_test_raw,  columns=cat_cols, drop_first=True).astype(float)
X_te = X_te.reindex(columns=X_tr.columns, fill_value=0)

corr_results = []
for col in X_tr.columns:
    r, p = pearsonr(X_tr[col], y_train)
    corr_results.append({'feature': col, 'r': r, '|r|': abs(r), 'p': p})
corr_df = pd.DataFrame(corr_results).sort_values('|r|', ascending=False)
print(corr_df.to_string())
print('\nStrongest correlation:', corr_df.iloc[0]['feature'])

---
## ❓ Q8.
> Run a linear regression using ALL features (statsmodels).
> **Which features are NOT statistically significant at alpha=0.05?**

In [ ]:
import statsmodels.api as sm

X_tr_const = sm.add_constant(X_tr)
model_full = sm.OLS(y_train, X_tr_const).fit()
print(model_full.summary())

print('\nNOT significant (p >= 0.05):')
print(model_full.pvalues[model_full.pvalues >= 0.05])

---
## ❓ Q9.
> Compute VIF for each feature.
> **Which features have VIF > 5?**

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

X_tr_c = sm.add_constant(X_tr)
vif_df = pd.DataFrame({
    'feature': X_tr_c.columns,
    'VIF': [variance_inflation_factor(X_tr_c.values, i) for i in range(X_tr_c.shape[1])]
})
print(vif_df.sort_values('VIF', ascending=False))
print('\nVIF > 5:')
print(vif_df[vif_df.VIF > 5])

---
## ❓ Q10.
> Run **Backward Variable Selection** (`forward=False`):
> - estimator: `LinearRegression()`
> - `k_features='best'`, `scoring='r2'`, `cv=5`
>
> **Which variables are selected?**

In [ ]:
from mlxtend.feature_selection import SequentialFeatureSelector as SFS

sfs_bwd = SFS(LinearRegression(), k_features='best',
              forward=False, floating=False, scoring='r2', cv=5)
sfs_bwd.fit(X_tr, y_train)
print('Backward selected:', sfs_bwd.k_feature_names_)

---
## ❓ Q11.
> Standardize features. Run `LassoCV(cv=5, random_state=1031)`.
> **Which features are zeroed out?**
> Using only non-zero features, fit a `LinearRegression`. **What is the TEST R²?**

In [ ]:
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score

scaler_fs = StandardScaler()
X_tr_sc = scaler_fs.fit_transform(X_tr)
X_te_sc = scaler_fs.transform(X_te)

lasso = LassoCV(cv=5, random_state=1031).fit(X_tr_sc, y_train)
coefs = pd.Series(lasso.coef_, index=X_tr.columns)
print('Lasso coefficients:'); print(coefs)
print('\nZeroed out:', coefs[coefs == 0].index.tolist())
selected = coefs[coefs != 0].index.tolist()
print('Selected:', selected)

# Fit LinearRegression with selected features
sel_idx = [list(X_tr.columns).index(s) for s in selected]
lr_lasso = LinearRegression().fit(X_tr_sc[:, sel_idx], y_train)
test_r2  = r2_score(y_test, lr_lasso.predict(X_te_sc[:, sel_idx]))
print('\nTest R² (Lasso-selected features):', test_r2)

---
## ❓ Q12.
> Apply PCA. **How many components retain ≥ 70% variance?**
> Fit `LinearRegression` with those components. **What is the TRAIN R²?**

In [ ]:
from sklearn.decomposition import PCA

pca_full = PCA().fit(X_tr_sc)
cumvar = np.cumsum(pca_full.explained_variance_ratio_)
print('Cumulative variance:')
for i, v in enumerate(cumvar, 1):
    print(f'  {i} components: {v:.4f}')

k = int(np.argmax(cumvar >= 0.70)) + 1
print(f'\nComponents for 70% variance: {k}')

pca_k = PCA(n_components=k)
X_tr_pca = pca_k.fit_transform(X_tr_sc)
X_te_pca = pca_k.transform(X_te_sc)

lr_pca = LinearRegression().fit(X_tr_pca, y_train)
print('Train R²:', r2_score(y_train, lr_pca.predict(X_tr_pca)))
print('Test  R²:', r2_score(y_test,  lr_pca.predict(X_te_pca)))

---
## Section 3: Decision Trees (REGRESSION)
*(Predict `Satisfaction` numeric. Use dummy-encoded features from Section 2.)*

---
## ❓ Q13.
> Fit a **Regression Tree** using only `FlightDistance`, `max_depth=2, random_state=1031`.
> Plot the tree.
> **What is the predicted `Satisfaction` for `FlightDistance = 500`?**

In [ ]:
from sklearn.tree import DecisionTreeRegressor, plot_tree
from sklearn.metrics import root_mean_squared_error
import matplotlib.pyplot as plt

tree_small = DecisionTreeRegressor(max_depth=2, random_state=1031)
tree_small.fit(X_train_raw[['FlightDistance']], y_train)

plt.figure(figsize=(12,5))
plot_tree(tree_small, feature_names=['FlightDistance'], filled=True, precision=2)
plt.show()

print('Predicted Satisfaction (FlightDistance=500):', tree_small.predict([[500]])[0])

---
## ❓ Q14.
> Fit a Regression Tree using ALL dummy-encoded features, `max_depth=4, random_state=1031`.
> **What is the TRAIN RMSE? What is the TEST RMSE?**

In [ ]:
tree4 = DecisionTreeRegressor(max_depth=4, random_state=1031)
tree4.fit(X_tr, y_train)

print('Tree (depth=4) Train RMSE:', root_mean_squared_error(y_train, tree4.predict(X_tr)))
print('Tree (depth=4) Test  RMSE:', root_mean_squared_error(y_test,  tree4.predict(X_te)))

---
## ❓ Q15.
> Fit a **MAXIMAL tree** (no constraints). **What is the TRAIN RMSE? TEST RMSE?**
> Is this evidence of overfitting?

In [ ]:
tree_max = DecisionTreeRegressor(random_state=1031)  # NO max_depth constraint
tree_max.fit(X_tr, y_train)

train_rmse_max = root_mean_squared_error(y_train, tree_max.predict(X_tr))
test_rmse_max  = root_mean_squared_error(y_test,  tree_max.predict(X_te))
print('Maximal Tree Train RMSE:', train_rmse_max)
print('Maximal Tree Test  RMSE:', test_rmse_max)
print('Gap (test-train):', test_rmse_max - train_rmse_max)
print('\nOverfitting? YES — train RMSE very low but test RMSE much higher')

---
## ❓ Q16.
> Tune with `GridSearchCV`, 5-fold CV, scoring = `neg_root_mean_squared_error`:
> `max_depth=[3,5,8,12]`, `min_samples_leaf=[5,20,100]`, `max_features=[4,8,None]`.
> **What is the optimal parameter combination?**

In [ ]:
from sklearn.model_selection import GridSearchCV

grid_tree = GridSearchCV(
    DecisionTreeRegressor(random_state=1031),
    {'max_depth': [3,5,8,12], 'min_samples_leaf': [5,20,100], 'max_features': [4,8,None]},
    cv=5, scoring='neg_root_mean_squared_error', n_jobs=-1
)
grid_tree.fit(X_tr, y_train)

print('Best params:', grid_tree.best_params_)
print('Best CV RMSE:', -grid_tree.best_score_)
print('Train RMSE:', root_mean_squared_error(y_train, grid_tree.predict(X_tr)))
print('Test  RMSE:', root_mean_squared_error(y_test,  grid_tree.predict(X_te)))

---
## Section 4: Ensemble Models (REGRESSION)
*(Predict `Satisfaction` numeric. Use dummy-encoded features.)*

---
## ❓ Q17.
> Fit a **Bagging Regressor** with 100 bootstrapped samples.
> Base estimator: `DecisionTreeRegressor(max_depth=10)`. `random_state=1031`.
> **What is the TRAIN RMSE?**

In [ ]:
from sklearn.ensemble import BaggingRegressor

bag = BaggingRegressor(
    estimator=DecisionTreeRegressor(max_depth=10),
    n_estimators=100, random_state=1031
)
bag.fit(X_tr, y_train)

print('Bagging Train RMSE:', root_mean_squared_error(y_train, bag.predict(X_tr)))
print('Bagging Test  RMSE:', root_mean_squared_error(y_test,  bag.predict(X_te)))

---
## ❓ Q18.
> Fit a **Random Forest Regressor** with 100 trees, `oob_score=True, random_state=1031`.
> **What is the OOB score?**

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(n_estimators=100, oob_score=True, random_state=1031)
rf.fit(X_tr, y_train)

print('RF OOB score:   ', rf.oob_score_)
print('RF Train RMSE:', root_mean_squared_error(y_train, rf.predict(X_tr)))
print('RF Test  RMSE:', root_mean_squared_error(y_test,  rf.predict(X_te)))

---
## ❓ Q19.
> Based on the Random Forest, **which feature is MOST IMPORTANT?**

In [ ]:
importances = pd.Series(rf.feature_importances_, index=X_tr.columns)
print(importances.sort_values(ascending=False))
print('\nMost important feature:', importances.idxmax())

---
## ❓ Q20.
> Fit a **Gradient Boosting Regressor** with 100 trees, `random_state=1031`.
> **What is the TRAIN RMSE?**

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

gb = GradientBoostingRegressor(n_estimators=100, random_state=1031)
gb.fit(X_tr, y_train)

print('GB Train RMSE:', root_mean_squared_error(y_train, gb.predict(X_tr)))
print('GB Test  RMSE:', root_mean_squared_error(y_test,  gb.predict(X_te)))

---
## ❓ Q21.
> Fit **Stochastic Gradient Boosting** with `n_estimators=100, max_features=0.4,
> subsample=0.6, random_state=1031`.
> **What is the TEST RMSE? Which feature is most important?**

In [ ]:
sgb = GradientBoostingRegressor(
    n_estimators=100, max_features=0.4, subsample=0.6, random_state=1031)
sgb.fit(X_tr, y_train)

print('SGB Train RMSE:', root_mean_squared_error(y_train, sgb.predict(X_tr)))
print('SGB Test  RMSE:', root_mean_squared_error(y_test,  sgb.predict(X_te)))

sgb_imp = pd.Series(sgb.feature_importances_, index=X_tr.columns)
print('\nMost important (SGB):', sgb_imp.idxmax())
print(sgb_imp.sort_values(ascending=False))

---
## Section 5: Support Vector Machines (SVR — Regression)
*(Predict `Satisfaction` numeric. Scale features.)*

---
## ❓ Q22.
> Scale all dummy-encoded features with `StandardScaler` (fit on train only).
> Fit `SVR(kernel='rbf')` with default parameters.
> **What is the TRAIN RMSE?**

In [ ]:
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler

scaler_svm = StandardScaler()
X_svm_tr = scaler_svm.fit_transform(X_tr)   # fit ONLY on train!
X_svm_te = scaler_svm.transform(X_te)

svr = SVR(kernel='rbf')  # default C=1, gamma='scale'
svr.fit(X_svm_tr, y_train)

print('SVR Train RMSE:', root_mean_squared_error(y_train, svr.predict(X_svm_tr)))
print('SVR Test  RMSE:', root_mean_squared_error(y_test,  svr.predict(X_svm_te)))

---
## ❓ Q23.
> Tune SVR with `GridSearchCV`, 5-fold CV, `scoring='neg_root_mean_squared_error'`:
> `C=[0.1,1,10,100]`, `gamma=[0.01,0.1,1]`.
> **What is the best C and gamma? What is the best CV score?**

In [ ]:
grid_svr = GridSearchCV(
    SVR(kernel='rbf'),
    {'C': [0.1, 1, 10, 100], 'gamma': [0.01, 0.1, 1]},
    cv=5, scoring='neg_root_mean_squared_error', n_jobs=-1
)
grid_svr.fit(X_svm_tr, y_train)

print('Best params:    ', grid_svr.best_params_)
print('Best CV RMSE:   ', -grid_svr.best_score_)

---
## ❓ Q24.
> Using the tuned SVR, **what is the TEST RMSE?**
> Compare all models by test RMSE. **Which model performs best?**

In [ ]:
svr_tuned_test = root_mean_squared_error(y_test, grid_svr.predict(X_svm_te))
print('Tuned SVR Test RMSE:', svr_tuned_test)

# Final comparison table
final = pd.DataFrame({
    'model': ['Linear Reg (full)', 'Reg Tree (tuned)', 'Random Forest',
              'Gradient Boosting', 'Stochastic GB', 'SVR (tuned)'],
    'test_rmse': [
        root_mean_squared_error(y_test, lr_pca.predict(X_te_pca)),   # PCA-based; swap for full LR if computed
        root_mean_squared_error(y_test, grid_tree.predict(X_te)),
        root_mean_squared_error(y_test, rf.predict(X_te)),
        root_mean_squared_error(y_test, gb.predict(X_te)),
        root_mean_squared_error(y_test, sgb.predict(X_te)),
        svr_tuned_test
    ]
}).set_index('model').sort_values('test_rmse')
print('\nFinal Model Comparison (Test RMSE):')
print(final)
print('\nBest model:', final.index[0])

---
# 📋 Quick Reference — Code Patterns (Set 3 Focus: Regression)

## Regression Tree vs Classification Tree
```python
# Regression: use DecisionTreeRegressor + root_mean_squared_error
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import root_mean_squared_error
tree = DecisionTreeRegressor(max_depth=4, random_state=1031)
tree.fit(X_tr, y_tr)                               # y_tr is NUMERIC
rmse = root_mean_squared_error(y_te, tree.predict(X_te))

# Classification: use DecisionTreeClassifier + accuracy_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
```

## Ensemble Regression
```python
from sklearn.ensemble import (
    BaggingRegressor, RandomForestRegressor,
    GradientBoostingRegressor
)
# Random Forest (Regression):
rf = RandomForestRegressor(n_estimators=100, oob_score=True, random_state=1031)
rf.oob_score_      # R² score based on out-of-bag predictions

# Stochastic Gradient Boosting:
sgb = GradientBoostingRegressor(
    n_estimators=100, max_features=0.4, subsample=0.6, random_state=1031)
```

## SVR (Support Vector Regression)
```python
from sklearn.svm import SVR
svr = SVR(kernel='rbf', C=1, gamma='scale')     # regression, not classification
svr.fit(X_tr_scaled, y_tr)                       # y_tr is numeric!
root_mean_squared_error(y_te, svr.predict(X_te_scaled))

# GridSearch for SVR (note scoring!)
GridSearchCV(SVR(), {'C': [...], 'gamma': [...]},
             cv=5, scoring='neg_root_mean_squared_error')
```

## ⚠️ Key Traps (Set 3 Specific)
| Trap | Rule |
|---|---|
| Regression tree metric | Use RMSE, NOT accuracy |
| Maximal tree | Train RMSE → 0, huge test RMSE = classic overfitting |
| GridSearch for regression | `scoring='neg_root_mean_squared_error'` (negative!) |
| SVR requires scaling | Same rule as SVC — always scale |
| OOB score | Only for RF/Bagging with `oob_score=True` |
| PCA threshold | `np.argmax(cumsum >= 0.70) + 1` for 70% |
| VIF threshold | > 5 concern; constant gets inf VIF (ignore it) |
| Step function reference | = FIRST label in labels list |
| Bagging `estimator=` | Use keyword `estimator` (not `base_estimator`) |